# Post-Training Usage: GNN Retrieval from Neo4j

Этот notebook показывает, как использовать GNN **после обучения**, когда `gnn_embedding` уже записаны в Neo4j.

Сценарий:
1. Получаем seed-сущности (из запроса или вручную).
2. Делаем graph expansion по `Entity.gnn_embedding` (поиск похожих сущностей).
3. Переходим к evidence через `(:Chunk)-[:MENTIONS]->(:Entity)`.
4. Возвращаем ранжированные чанки с цитатами (`doc_id`, `page`, `span`).


In [ ]:
# При необходимости:
# %pip install neo4j numpy pandas


In [ ]:
import os
import re
from collections import defaultdict

import numpy as np
import pandas as pd
from neo4j import GraphDatabase


In [ ]:
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'neo4j_password')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(query: str, params: dict | None = None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [r.data() for r in result]

health = run_query("""
MATCH (e:Entity)
WITH count(e) AS entities
MATCH (c:Chunk)
RETURN entities, count(c) AS chunks
""")
health


In [ ]:
# Загружаем только сущности, у которых уже есть gnn_embedding
entity_rows = run_query("""
MATCH (e:Entity)
WHERE e.gnn_embedding IS NOT NULL
RETURN
  e.entity_id AS entity_id,
  coalesce(e.entity_name, e.normalized_value, e.value, '') AS entity_text,
  e.gnn_embedding AS gnn_embedding
""")

entity_df = pd.DataFrame(entity_rows)
if entity_df.empty:
    raise ValueError('В базе нет Entity.gnn_embedding. Сначала выполните training notebook и upsert эмбеддингов.')

entity_df = entity_df.dropna(subset=['entity_id', 'gnn_embedding']).drop_duplicates('entity_id').reset_index(drop=True)
entity_matrix = np.vstack(entity_df['gnn_embedding'].apply(lambda x: np.array(x, dtype=np.float32)).values)

# L2-нормализация для cosine = dot
norm = np.linalg.norm(entity_matrix, axis=1, keepdims=True) + 1e-12
entity_matrix = entity_matrix / norm

id_to_idx = {eid: i for i, eid in enumerate(entity_df['entity_id'])}
idx_to_id = {i: eid for eid, i in id_to_idx.items()}

print('entities with gnn_embedding:', len(entity_df))
print('embedding dim:', entity_matrix.shape[1])


In [ ]:
def topk_similar_entities_by_seed(seed_entity_ids: list[str], top_k: int = 50):
    """
    Graph expansion: берём seed entities и находим ближайшие по gnn_embedding.
    Возвращает DataFrame: entity_id, entity_text, score.
    """
    valid_seeds = [eid for eid in seed_entity_ids if eid in id_to_idx]
    if not valid_seeds:
        return pd.DataFrame(columns=['entity_id', 'entity_text', 'score'])

    seed_vectors = entity_matrix[[id_to_idx[eid] for eid in valid_seeds]]
    query_vec = seed_vectors.mean(axis=0)
    query_vec = query_vec / (np.linalg.norm(query_vec) + 1e-12)

    scores = entity_matrix @ query_vec
    top_idx = np.argsort(-scores)[:top_k]

    out = entity_df.iloc[top_idx][['entity_id', 'entity_text']].copy()
    out['score'] = scores[top_idx]
    return out.reset_index(drop=True)

def lexical_seed_entities(query: str, top_k: int = 10):
    """
    Простой baseline для выбора seed entities из текстового запроса.
    Без fulltext индекса: через token overlap в Python.
    """
    tokens = [t for t in re.split(r'[^\w]+', query.lower()) if len(t) > 1]
    if not tokens:
        return pd.DataFrame(columns=['entity_id', 'entity_text', 'lex_score'])

    base = entity_df[['entity_id', 'entity_text']].copy()
    text_l = base['entity_text'].fillna('').str.lower()

    scores = np.zeros(len(base), dtype=np.float32)
    for t in tokens:
        scores += text_l.str.contains(re.escape(t), regex=True).astype(np.float32).values

    base['lex_score'] = scores
    base = base[base['lex_score'] > 0].sort_values('lex_score', ascending=False).head(top_k)
    return base.reset_index(drop=True)

def fetch_chunks_for_entities(entity_scores_df: pd.DataFrame, max_chunks: int = 30):
    """
    Переход Entity -> Chunk через MENTIONS + агрегированный GNN score чанка.
    """
    if entity_scores_df.empty:
        return pd.DataFrame(columns=['chunk_id', 'doc_id', 'page', 'span_start', 'span_end', 'text', 'gnn_score'])

    rows = entity_scores_df[['entity_id', 'score']].to_dict('records')
    chunk_rows = run_query("""
    UNWIND $rows AS row
    MATCH (c:Chunk)-[:MENTIONS]->(e:Entity {entity_id: row.entity_id})
    RETURN
      c.chunk_id AS chunk_id,
      c.doc_id AS doc_id,
      c.page AS page,
      c.span_start AS span_start,
      c.span_end AS span_end,
      coalesce(c.text, '') AS text,
      row.score AS entity_score
    """, {'rows': rows})

    if not chunk_rows:
        return pd.DataFrame(columns=['chunk_id', 'doc_id', 'page', 'span_start', 'span_end', 'text', 'gnn_score'])

    cdf = pd.DataFrame(chunk_rows)
    agg = cdf.groupby(['chunk_id', 'doc_id', 'page', 'span_start', 'span_end', 'text'], as_index=False)['entity_score'].sum()
    agg = agg.rename(columns={'entity_score': 'gnn_score'})
    agg = agg.sort_values('gnn_score', ascending=False).head(max_chunks).reset_index(drop=True)
    return agg


In [ ]:
def retrieve_with_gnn(query: str, seed_top_k: int = 10, expand_top_k: int = 60, out_top_k: int = 20):
    """
    Полный post-training retrieval pipeline:
    query -> lexical seed entities -> GNN expansion -> chunks
    """
    seeds = lexical_seed_entities(query, top_k=seed_top_k)
    if seeds.empty:
        return {
            'query': query,
            'seed_entities': seeds,
            'expanded_entities': pd.DataFrame(),
            'chunks': pd.DataFrame(),
        }

    expanded = topk_similar_entities_by_seed(seeds['entity_id'].tolist(), top_k=expand_top_k)
    chunks = fetch_chunks_for_entities(expanded, max_chunks=out_top_k)

    return {
        'query': query,
        'seed_entities': seeds,
        'expanded_entities': expanded,
        'chunks': chunks,
    }


In [ ]:
# Пример 1: retrieval от текстового запроса
query = 'ответственность работодателя за задержку зарплаты'
result = retrieve_with_gnn(query, seed_top_k=10, expand_top_k=60, out_top_k=20)

print('Query:', result['query'])
print('Seed entities:', len(result['seed_entities']))
print('Expanded entities:', len(result['expanded_entities']))
print('Top chunks:', len(result['chunks']))

display(result['seed_entities'].head(10))
display(result['expanded_entities'].head(15))
display(result['chunks'][['chunk_id', 'doc_id', 'page', 'span_start', 'span_end', 'gnn_score']].head(20))


In [ ]:
# Пример 2: retrieval от конкретной seed entity (если знаете entity_id)
# seed_entity_id = '010dca0a987959b93d66c4a456e31a0f221ad294d3370769db1f9be4848e2ca1'
# expanded = topk_similar_entities_by_seed([seed_entity_id], top_k=50)
# chunks = fetch_chunks_for_entities(expanded, max_chunks=20)
# display(expanded.head(15))
# display(chunks[['chunk_id', 'doc_id', 'page', 'span_start', 'span_end', 'gnn_score']].head(20))


In [ ]:
# Опционально: как добавить fusion с BM25/semantic, если у вас есть эти скоры
# chunks['bm25_norm'] = ...
# chunks['semantic_norm'] = ...
# chunks['gnn_norm'] = (chunks['gnn_score'] - chunks['gnn_score'].min()) / (chunks['gnn_score'].max() - chunks['gnn_score'].min() + 1e-12)
# chunks['fusion'] = 0.45 * chunks['bm25_norm'] + 0.35 * chunks['gnn_norm'] + 0.20 * chunks['semantic_norm']
# chunks = chunks.sort_values('fusion', ascending=False)


In [ ]:
driver.close()
